# Dự án Nhận diện Thủ ngữ (Sign Language Recognition) bằng YOLO26

## 1. Giới thiệu dự án
Dự án này tập trung vào việc xây dựng và huấn luyện một mô hình học sâu để nhận diện các ký hiệu thủ ngữ (Sign Language). Mô hình được sử dụng trong notebook này là **YOLO26**, nhằm mục đích phát hiện và phân loại các cử chỉ tay theo thời gian thực với tốc độ và độ chính xác cao.

## 2. Mục tiêu
- Tiền xử lý và chuẩn bị tập dữ liệu (dataset) thủ ngữ để đưa vào huấn luyện.
- Khởi tạo và cấu hình mô hình nhận diện YOLO26.
- Huấn luyện (Train) mô hình để nhận diện chính xác các từ vựng hoặc chữ cái thủ ngữ.
- Đánh giá (Evaluate) hiệu suất của mô hình thông qua các độ đo cơ bản như mAP, Precision, Recall.
- Thử nghiệm (Inference) mô hình trên hình ảnh hoặc video thực tế.

## 3. Cấu trúc Notebook
1. **Cài đặt và Import thư viện:** Thiết lập môi trường và tải các thư viện cần thiết (ví dụ: opencv, pytorch, matplotlib, các thư viện hỗ trợ YOLO26...).
2. **Khám phá và Tiền xử lý dữ liệu (EDA):** Tải bộ dữ liệu, kiểm tra số lượng ảnh, label format, và phân chia train/val/test.
3. **Cấu hình YOLO26:** Tạo file `data.yaml` cấu hình đường dẫn và thông tin số lượng class (nhãn dán) cho thủ ngữ.
4. **Huấn luyện mô hình (Training):** Thực thi tiến trình train YOLO26 trên dữ liệu đã chuẩn bị.
5. **Kiểm tra và Trực quan hóa (Validation & Visualization):** Xem các biểu đồ loss, metrics và so sánh kết quả ground-truth với dự đoán (predictions).

## 4. Cài đặt môi trường (Colab + GPU)
Bài toán: **Object Detection** ký hiệu tay bằng **YOLO26** (fine-tune từ 80 lớp COCO sang số lớp
của dataset). Cài `ultralytics` + `roboflow`, kiểm tra GPU.

In [1]:
!pip install -q ultralytics roboflow

import ultralytics, torch
ultralytics.checks()
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.5/107.7 GB disk)
Torch: 2.11.0+cpu | CUDA: False
GPU: CPU only


## 5. Tải dataset từ Roboflow
Dataset detection ảnh thật (có bounding box, format YOLO). API key nhập lúc chạy bằng `getpass`
(không lưu vào file). Lấy key tại Roboflow → *Settings* → *API Keys*.

In [4]:
from roboflow import Roboflow
from getpass import getpass

rf = Roboflow(api_key=getpass("Roboflow API key: "))
project = rf.workspace("pinn-b18pct-gmail-com").project("sign-language-sokdr-vckos")
versions = project.versions()
print("Các version có sẵn:", [v.version for v in versions])

latest = versions[0]                      # bản mới nhất
dataset = latest.download("yolov8")
print("Đã tải:", latest.version, "->", dataset.location)


Roboflow API key: ··········
loading Roboflow workspace...
loading Roboflow project...
Các version có sẵn: ['1']
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to sign-language-1 in yolov8:: 100%|██████████| 1445/1445 [00:03<00:00, 409.63it/s] 


Đã tải: 1 -> /content/sign-language-1


## 6. Chuẩn hoá `data.yaml`
`data.yaml` của Roboflow hay dùng đường dẫn tương đối dễ lỗi trên Colab → ghi lại bằng path tuyệt đối.

In [5]:
import yaml
from pathlib import Path

loc = Path(dataset.location)
DATA_YAML = loc / "data.yaml"
cfg = yaml.safe_load(DATA_YAML.read_text())

cfg["path"] = str(loc)
for key, candidates in [("train", ["train/images"]),
                        ("val", ["valid/images", "val/images"]),
                        ("test", ["test/images"])]:
    for c in candidates:
        if (loc / c).exists():
            cfg[key] = c
            break

DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
DATA_YAML = str(DATA_YAML)
print("data.yaml đã chuẩn hoá:\n")
print(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

data.yaml đã chuẩn hoá:

names:
- A
- B
- C
- D
- E
- F
- G
- H
- I
- J
- K
- L
- M
- N
- O
- P
- Q
- R
- S
- T
- U
- V
- W
- X
- Y
- Z
nc: 26
roboflow:
  license: CC BY 4.0
  project: sign-language-sokdr-vckos
  url: https://universe.roboflow.com/pinn-b18pct-gmail-com/sign-language-sokdr-vckos/dataset/1
  version: 1
  workspace: pinn-b18pct-gmail-com
test: test/images
train: train/images
val: valid/images
path: /content/sign-language-1



## 7. Khám phá dữ liệu (EDA)
Kiểm tra số lớp và số ảnh từng split.

In [6]:
from pathlib import Path

print("Số lớp (nc):", cfg.get("nc"))
print("Tên lớp:", cfg.get("names"))

exts = {".jpg", ".jpeg", ".png", ".bmp"}
for key in ["train", "val", "test"]:
    if key in cfg:
        d = Path(cfg["path"]) / cfg[key]
        n = sum(1 for x in d.glob("*") if x.suffix.lower() in exts)
        print(f"{key}: {n} ảnh  ({d})")

Số lớp (nc): 26
Tên lớp: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
train: 504 ảnh  (/content/sign-language-1/train/images)
val: 144 ảnh  (/content/sign-language-1/valid/images)
test: 72 ảnh  (/content/sign-language-1/test/images)


## 8. Khởi tạo mô hình YOLO26n (detection)
Tải pretrained 80 lớp COCO làm điểm khởi đầu. Khi train, Ultralytics **tự đổi detection head**
từ 80 lớp → đúng `nc` của dataset (đây chính là phần "đơn giản hoá 80 → N class").

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")   # detection, nano
model.info()

## 9. Huấn luyện (Training)
Fine-tune trên dataset ký hiệu tay. Tự chọn GPU nếu có.

In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=DEVICE,
    patience=20,                 # early stopping
    project="runs/sign_language",
    name="yolo26n_det",
)
print("Kết quả lưu tại:", results.save_dir)
print("Best weights:", Path(results.save_dir) / "weights" / "best.pt")

## 9b. (Tùy chọn) Train với FREEZE BACKBONE để so sánh
Đóng băng 10 module đầu (backbone) → chỉ train phần head. Phù hợp dataset nhỏ: giảm overfit, train nhanh.
Tạo model riêng (`model_frozen`) để không đè model full fine-tune ở trên.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model_frozen = YOLO("yolo26n.pt")
results_frozen = model_frozen.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    device=DEVICE,
    patience=30,
    freeze=10,                   # đóng băng backbone (module 0-9), chỉ train head
    project="runs/sign_language",
    name="yolo26n_frozen",
)
print("Frozen best:", Path(results_frozen.save_dir) / "weights" / "best.pt")

In [ ]:
# So sánh mAP: full fine-tune vs freeze backbone
m_full = model.val()
m_frozen = model_frozen.val()
print(f"Full fine-tune  | mAP50={m_full.box.map50:.4f}  mAP50-95={m_full.box.map:.4f}")
print(f"Freeze backbone | mAP50={m_frozen.box.map50:.4f}  mAP50-95={m_frozen.box.map:.4f}")
print("=> Giữ model có mAP50 cao hơn cho webcam (best.pt tương ứng).")

## 10. Đánh giá (mAP)

In [ ]:
metrics = model.val()
print("mAP50-95:", round(metrics.box.map, 4))
print("mAP50   :", round(metrics.box.map50, 4))
print("Precision:", round(metrics.box.mp, 4), "| Recall:", round(metrics.box.mr, 4))

## 11. Trực quan hoá kết quả train

In [ ]:
from IPython.display import Image, display
from pathlib import Path

save_dir = Path(results.save_dir)
for name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg", "BoxF1_curve.png"]:
    p = save_dir / name
    if p.exists():
        print(p)
        display(Image(str(p)))

## 12. Inference thử trên ảnh val

In [ ]:
from pathlib import Path
from IPython.display import Image, display

val_dir = Path(cfg["path"]) / cfg["val"]
sample = next(p for p in val_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

pred = model.predict(sample, conf=0.5, device=DEVICE, save=True)[0]
print("Ảnh:", sample)
for b in pred.boxes:
    print(f"  {pred.names[int(b.cls)]}  conf={float(b.conf):.2f}  bbox={[round(v, 1) for v in b.xyxy.tolist()[0]]}")

out_img = Path(pred.save_dir) / sample.name
if out_img.exists():
    display(Image(str(out_img)))

## 13. Tải `best.pt` về máy để chạy webcam local
**Webcam KHÔNG chạy trên Colab** (Colab không truy cập camera laptop). Tải `best.pt` về,
rồi chạy script webcam ở máy local.

In [ ]:
from google.colab import files
from pathlib import Path

best = str(Path(results.save_dir) / "weights" / "best.pt")
print("Tải file:", best)
files.download(best)

### Script chạy webcam ở MÁY LOCAL (không chạy trong Colab)
Cài ở local: `pip install ultralytics opencv-python`. Đặt `best.pt` cùng thư mục rồi chạy:

```python
from ultralytics import YOLO

model = YOLO("best.pt")              # model đã fine-tune
model.predict(source=0, show=True, conf=0.5)   # source=0 = webcam; nhấn q để thoát
```